# Qwen 前向传播：从文本到下一个 Token

这份 Notebook 沿着真实模型的数据路径，观察文本如何变成 Token ID、隐藏表示和下一 Token 的预测。讲解延续 MiniGPT Notebook 的方式：先明确模块作用，再对应实际变量、函数调用和张量维度。

学习顺序：**加载 → 文本编码 → Embedding → 完整前向与 Logits → 选择下一个 Token**。

统一记号：`B` 为 Batch，`T` 为当前输入长度，`D` 为隐藏维度，`V` 为模型输出词表维度。本例保存的值为 `B=1, T=7, D=896, V=151936`。`input_ids` 只有两个轴，第三个 Hidden Dimension 在 Embedding 后才出现。

下文的实际数值均来自原有输出；没有打印的中间维度根据代码推导。本次仅补充 Markdown，没有执行 Notebook，也没有更新任何原有输出。

## 1. 加载模型与 Tokenizer

这一模块建立两个配套对象：处理文本的 `tokenizer`，以及执行神经网络计算的 `model`。

- `model_name = "Qwen/Qwen2.5-0.5B"` 指定预训练基础语言模型；这里没有使用 Instruct 模型或对话模板。
- `AutoTokenizer.from_pretrained(model_name)` 读取配套分词规则、词表和配置，供文本编码与 Token ID 解码使用。
- `AutoModelForCausalLM.from_pretrained(model_name)` 根据配置选择具体模型类，并加载预训练权重。`CausalLM` 表示带语言模型预测头的因果语言模型。
- `from_pretrained()` 从本地缓存或模型仓库读取资源，不是在这里训练模型。
- `model.eval()` 切换评估模式；它不会自动关闭梯度记录，后面的 `torch.no_grad()` 才负责这一点。

**Qwen** 提供模型及配套资源，**Tokenizer** 负责文本与 ID 转换，**Transformers** 提供加载接口与模型实现，**PyTorch** 承担张量计算。

最后一行 `model.eval()` 返回模型对象，因此 Jupyter 显示了模型结构。已有输出是外层 `Qwen2ForCausalLM` 包含主体 `Qwen2Model` 和 `lm_head`；主体中有 `Embedding(151936, 896)` 与 24 层 Decoder。这些维度将在后面的实际张量中逐一对应。

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## 2. 文本编码与输入张量

`tokenizer(text, return_tensors="pt")` 将字符串分词、转换为整数 ID，并返回 PyTorch 张量。`inputs` 是包含多个字段的映射对象，不是已经完成 Embedding 的特征张量。

已有输出中，`"Hello, I am a student."` 编码为：

`[9707, 11, 358, 1079, 264, 5458, 13]`

因此序列长度是 **7 个 Token**。它不是按空格数出的英文单词数量；标点、空格与子词如何编码取决于当前 Tokenizer，其他文本应以实际编码长度为准。

| 变量 | 本例 shape | 含义 |
|---|---|---|
| `input_ids` | `[1, 7] = [B, T]` | 1 条文本，7 个整数 ID |
| `attention_mask` | `[1, 7] = [B, T]` | 输入位置是否有效，通常 1 表示有效，0 表示 padding |

本例没有 padding，所以 `attention_mask` 全为 1。它与因果 Attention 的三角遮罩职责不同：全 1 只说明所有输入位置有效，不意味着允许当前位置看到未来 Token。

Batch 表示一次处理多少条序列，Sequence 表示每条序列有多少个 Token；Hidden Dimension 表示每个位置的特征数。此处尚无 Hidden 轴，下一节 Embedding 后才成为 `[B, T, D]`。

In [2]:
text = "Hello, I am a student."
inputs = tokenizer(text, return_tensors="pt")

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

print("input_ids:")
print(input_ids)

print("\ninput_ids shape:")
print(input_ids.shape)

print("\nattention_mask:")
print(attention_mask)

print("\nattention_mask shape:")
print(attention_mask.shape)

input_ids:
tensor([[9707,   11,  358, 1079,  264, 5458,   13]])

input_ids shape:
torch.Size([1, 7])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1]])

attention_mask shape:
torch.Size([1, 7])


## 3. Token Embedding

`model.model.embed_tokens(input_ids)` 直接调用主体的词嵌入层，从 `[151936, 896]` 的权重矩阵中按 Token ID 查找对应行。

| 阶段 | shape | 每个位置的内容 |
|---|---|---|
| 输入 `input_ids` | `[1, 7]` | 一个整数索引 |
| 输出 `embeddings` | `[1, 7, 896]` | 一个 896 维的浮点向量 |

Batch 和 Sequence 两个轴保持不变；新出现的 `896` 来自模型的 `hidden_size`。这里是每个 Token 的**初始表示**，还没有经过 Decoder 层融合上下文。

`torch.no_grad()` 关闭这段计算的梯度记录，减少自动求导开销；它不会更新权重，也不能代替 `model.eval()`。

与 MiniGPT 中给 Embedding 加正弦位置向量的实现不同，Qwen 使用 RoPE，在 Attention 的 Q/K 中引入位置信息；单独调用这一行 Embedding 尚未执行 RoPE。已有输出确认 `embeddings.shape = [1, 7, 896]`。

In [3]:
with torch.no_grad():
    embeddings = model.model.embed_tokens(input_ids)

print("embedding shape:")
print(embeddings.shape)

embedding shape:
torch.Size([1, 7, 896])


## 4. 前向传播与 Logits

`model(input_ids=input_ids, attention_mask=attention_mask)` 调用带预测头的完整模型。它重新从 `input_ids` 开始计算，**没有直接使用上一节的变量 embeddings**。

```text
input_ids [1, 7]
    ↓ Token Embedding
hidden states [1, 7, 896]
    ↓ 24 层 Decoder（含 RoPE、因果 Attention、MLP 与残差）
    ↓ 主体最终 RMSNorm
final hidden states [1, 7, 896]
    ↓ lm_head
logits [1, 7, 151936]
```

`outputs` 是结构化返回对象；`outputs.logits` 才是当前要观察的张量。`logits[b, t, :]` 表示样本 `b` 在位置 `t`、基于截至该位置的上下文，对下一个 Token 给出的词表分数。

- **896** 是每个位置的内部隐藏表示维度。
- **151936** 是 `model.config.vocab_size`，也就是预测头的输出通道数；不能理解为 151936 个自然语言单词。
- **Logits 是未归一化分数，不是概率**。若要得到概率分布，应沿词表轴做 Softmax。

已有输出 `[1, 7, 151936]` 表明一次前向返回了全部 7 个位置的预测。这里没有传入训练标签、没有计算训练 loss，也没有执行连续生成循环。

In [4]:
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

logits = outputs.logits

print("logits shape:")
print(logits.shape)

logits shape:
torch.Size([1, 7, 151936])


### 4.1 用输入与配置交叉核对维度

下面的 Cell 读取已经存在的输入和模型配置，不会再次执行前向：

- `batch_size`、`sequence_length` 从 `input_ids.shape` 读取，取决于当前输入。
- `hidden_size`、`vocab_size` 从 `model.config` 读取，属于模型配置。

因此预期 Embedding shape 为 `(batch_size, sequence_length, hidden_size) = (1, 7, 896)`，预期 Logits shape 为 `(batch_size, sequence_length, vocab_size) = (1, 7, 151936)`。

已有打印与前面保存的实际张量形状一致。改变文本长度通常会改变 `T`，不会改变这个模型的 `D` 和 `V`。

In [5]:
batch_size = input_ids.shape[0]
sequence_length = input_ids.shape[1]
hidden_size = model.config.hidden_size
vocab_size = model.config.vocab_size

print("batch_size:", batch_size)
print("sequence_length:", sequence_length)
print("hidden_size:", hidden_size)
print("vocab_size:", vocab_size)

print("\nExpected embedding shape:")
print((batch_size, sequence_length, hidden_size))

print("\nExpected logits shape:")
print((batch_size, sequence_length, vocab_size))

batch_size: 1
sequence_length: 7
hidden_size: 896
vocab_size: 151936

Expected embedding shape:
(1, 7, 896)

Expected logits shape:
(1, 7, 151936)


## 5. 从 Logits 预测下一个 Token

要续写整个输入，使用最后一个位置对下一 Token 的预测：

| 表达式或变量 | 本例 shape | 作用 |
|---|---|---|
| `logits` | `[1, 7, 151936]` | 全部样本、全部位置的词表分数 |
| `logits[:, -1, :]`，即 `last_logits` | `[1, 151936]` | 保留 Batch，只取最后位置 |
| `torch.argmax(last_logits, dim=-1)` | `[1]` | 每个样本选择词表分数最高的 ID |
| `tokenizer.decode(next_token_id)` | 字符串 | 将选中的 ID 解码为文本片段 |

`-1` 索引的是 Sequence 轴的最后位置，不是词表末尾。该位置已通过因果 Attention 汇集前面的可见上下文，**读取最后位置不等于只使用最后一个输入 Token**。前面位置的 Logits 对应各自较短前缀之后的预测。

这里是贪心选择；Softmax 保持候选分数的大小顺序，所以仅取最大项时不必先计算概率。

已有输出选择 ID `358`，解码为 `' I'`，包含前导空格。当前 Cell 只预测一次，没有将它拼回输入，也没有继续生成后续文本。

In [6]:
last_logits = logits[:, -1, :]

next_token_id = torch.argmax(last_logits, dim=-1)
next_token = tokenizer.decode(next_token_id)

print("next token id:")
print(next_token_id)

print("\nnext token:")
print(next_token)

next token id:
tensor([358])

next token:
 I
